<b>Running inference with GLiREL models</b>

This notebook shows how to generate RE predictions between named entities aligned with the ArchOnto ontology from datasets available at: [https://doi.org/10.6084/m9.figshare.28970633](https://doi.org/10.6084/m9.figshare.28970633).

<b>Import the required libraries</b>

In [ ]:
from glirel import GLiREL
import json

<b>Define and parse mapping rules between entity descriptors and ArchOnto classes</b>

In [ ]:
label_mapping = {"E21":"Person", "E74":"Organization", "E53":"Place", "ARE2":"Title", "ARE8":"Role", "E5":"Event", "E52":"Date", "E54":"Dimension"}

# Replace ArchOnto classes with entity descriptors
def replace_ner_types(ner_array, tag_dict):
    result = []
    for item in ner_array:
        new_item = item.copy()
        tag_key = item[2]
        new_item[2] = tag_dict.get(tag_key, tag_key)
        result.append(new_item)
    return result

<b>Define head and tail relation constraints</b>

In [ ]:
constraints = {
        "no relation": {},
        # Relations related to events
        "has time-span":{"allowed_head": ["Event"], "allowed_tail": ["Date"]},
        "took place at":{"allowed_head": ["Event"], "allowed_tail": ["Place"]},
        # Relations related to works/documents
        "author of":{"allowed_head": ["Person"], "allowed_tail": ["Title"]},
        "date of creation":{"allowed_head": ["Title"], "allowed_tail": ["Date"]},
        "editor of":{"allowed_head": ["Person"], "allowed_tail": ["Title"]},
        "produced by":{"allowed_head": ["Title"], "allowed_tail": ["Person",'Role']},  
        "in the role of":{"allowed_head": ["Person","Organization"], "allowed_tail": ["Role"]},
        # Relations related to role types
        "has owner":{"allowed_head": ["Organization"], "allowed_tail": ["Person","Role"]},
        # Relations between Group/Place
        "has current or former location":{"allowed_head": ["Organization"], "allowed_tail": ["Place"]},
        "has current or former residence":{"allowed_head": ["Organization"], "allowed_tail": ["Place"]},
        # Relations related to Place
        "falls within":{"allowed_head": ["Place"], "allowed_tail": ["Place"]},
        # Relations between Person/Group
        "has current or former member":{"allowed_head": ["Organization"], "allowed_tail": ["Person","Organization"]},
        "director of":{"allowed_head": ["Person"], "allowed_tail": ["Organization"]},
        "participant in":{"allowed_head": ["Organization"], "allowed_tail": ["Person",'Role']},
}
relation_labels=constraints.keys()

<b>Load ground-truth files with annotated entities and relations in JSONL format</b>

In [ ]:
# Dataset options https://doi.org/10.6084/m9.figshare.28970633
with open('re/datasets/re-domain-gen/test.jsonl', 'r') as file:
    data = json.load(file)

text=data["text"]
tokens=text.split(" ")
ner=data["ner"]
mapped_ner=replace_ner_types(ner, label_mapping)

<b>Load and execute GLiREL model</b>

In [ ]:
# Glirel model https://huggingface.co/jackboyla/glirel-large-v0
glirel_model = 'jackboyla/glirel-large-v0'

print(f"Loading GLiREL model: {glinrel_model}")
model = GLiREL.from_pretrained(glirel_model)

# Predict relations (you can adjust threshold and top_k as needed)
predictions = model.predict_relations(tokens, constraints, threshold=0.5, ner=mapped_ner, top_k=1)
# OR predict relations without constraints
predictions = model.predict_relations(tokens, relation_labels, threshold=0.5, ner=mapped_ner, top_k=1)